# Radiant RAG MCP Server — Test Notebook

Tests all ten MCP tools exposed by `radiant-rag-mcp` against a live server instance.

| | |
|---|---|
| **Transport** | Streamable HTTP (server in background thread, client on localhost) |
| **Backend** | ChromaDB (default — no external services) |
| **LLM** | Ollama Cloud `gemma4:31b-cloud` via `https://ollama.com/v1` |
| **LLM tools** | Cells marked `[LLM]` require `RADIANT_OLLAMA_API_KEY` — skipped automatically if absent |

**Before running:** create an API key at https://ollama.com/settings/keys and add it to
Colab Secrets (key icon in the left sidebar) as `OLLAMA_API_KEY`.

---

## Sections
1. Install
2. Import check
3. Configuration
4. Helpers
5. Server startup
6. Connection verification
7. Pre-ingestion state
8. Document ingestion
9. Search (no LLM)
10. Query `[LLM]`
11. Multi-turn conversation `[LLM]`
12. Maintenance
13. Summary


## 1  Install


In [1]:
import subprocess, sys

ACTIVE_BRANCH = 'mcp'
CONFIG_PATH   = '/content/config.yaml'

# ── Install ───────────────────────────────────────────────────────────────────
!pip install -q "radiant-rag-mcp[chroma] @ git+https://github.com/dshipley71/radiant-rag.git@mcp"
!pip install -q --prefer-binary nest_asyncio httpx fastmcp
!wget -q "https://raw.githubusercontent.com/dshipley71/radiant-rag/mcp/config.yaml" -O {CONFIG_PATH}

# ── Pre-cache model weights ───────────────────────────────────────────────────
# Both models are loaded by the server at startup. Caching them here means
# startup is instantaneous instead of waiting for ~175 MB of downloads.
!python3 -c "from sentence_transformers import SentenceTransformer; \
    SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2'); \
    print('sentence-transformers  ✓')"
!python3 -c "from sentence_transformers import CrossEncoder; \
    CrossEncoder('cross-encoder/ms-marco-MiniLM-L12-v2'); \
    print('cross-encoder          ✓')"

print('Install complete  ✓')


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 17.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 40.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 707.3/707.3 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6

## 2  Import check


In [2]:
import sys

_missing = []

# Check the MCP package itself
try:
    import radiant_rag_mcp
    print(f'  ✓  radiant_rag_mcp  ({radiant_rag_mcp.__file__})')
except ImportError:
    print('  ✗  radiant_rag_mcp  — re-run the install cell')
    _missing.append('radiant-rag-mcp (re-run install cell)')

# Check runtime dependencies
checks = [
    ('fastmcp',               'fastmcp'),
    ('nest_asyncio',          'nest_asyncio'),
    ('chromadb',              'chromadb'),
    ('sentence_transformers', 'sentence-transformers'),
    ('openai',                'openai'),
    ('yaml',                  'pyyaml'),
    ('httpx',                 'httpx'),
]

for module, pkg in checks:
    try:
        __import__(module)
        print(f'  ✓  {module}')
    except ImportError:
        print(f'  ✗  {module}  →  pip install --prefer-binary {pkg}')
        _missing.append(pkg)

print()
if _missing:
    print(f'[ACTION] Missing: {", ".join(_missing)}')
    print('         Fix the install cell and re-run before continuing.')
else:
    print('All imports resolved  ✓  — safe to continue')


  ✓  radiant_rag_mcp  (/usr/local/lib/python3.12/dist-packages/radiant_rag_mcp/__init__.py)
  ✓  fastmcp
  ✓  nest_asyncio
  ✓  chromadb
  ✓  sentence_transformers
  ✓  openai
  ✓  yaml
  ✓  httpx

All imports resolved  ✓  — safe to continue


## 3  Configuration

Set `RADIANT_OLLAMA_API_KEY` to enable LLM tools.
Get a key from https://ollama.com/settings/keys


In [3]:
import os
from google.colab import userdata

# ── Ollama Cloud ──────────────────────────────────────────────────────────────
# Base URL: HTTPS required, /v1 required (OpenAI-compatible API path).
# For local Ollama: 'http://localhost:11434/v1'
os.environ['RADIANT_OLLAMA_BASE_URL'] = 'https://ollama.com/v1'
os.environ['RADIANT_OLLAMA_API_KEY']  = userdata.get('OLLAMA_API_KEY')

# ── Transport ─────────────────────────────────────────────────────────────────
# HTTP transport: server and client coexist in the same Colab process.
os.environ['RADIANT_TRANSPORT'] = 'http'
os.environ['RADIANT_HOST']      = '127.0.0.1'
os.environ['RADIANT_PORT']      = '8765'

# ── Storage ───────────────────────────────────────────────────────────────────
os.environ['RADIANT_STORAGE_BACKEND'] = 'chroma'

# ── Config file ───────────────────────────────────────────────────────────────
os.environ['RADIANT_CONFIG_PATH'] = CONFIG_PATH

SERVER_URL  = f"http://{os.environ['RADIANT_HOST']}:{os.environ['RADIANT_PORT']}/mcp"
HAS_LLM_KEY = bool(os.environ.get('RADIANT_OLLAMA_API_KEY'))

print(f'Branch        : {ACTIVE_BRANCH}')
print(f'Ollama URL    : {os.environ["RADIANT_OLLAMA_BASE_URL"]}')
print(f'LLM key set   : {HAS_LLM_KEY}')
print(f'Server URL    : {SERVER_URL}')
print(f'Backend       : {os.environ["RADIANT_STORAGE_BACKEND"]}')
print(f'Config        : {CONFIG_PATH}')

if not HAS_LLM_KEY:
    print()
    print('[NOTE] No API key — LLM cells will be skipped.')
    print('       Get a key at https://ollama.com/settings/keys')


Branch        : mcp
Ollama URL    : https://ollama.com/v1
LLM key set   : True
Server URL    : http://127.0.0.1:8765/mcp
Backend       : chroma
Config        : /content/config.yaml


## 4  Helpers


In [4]:
import asyncio
import json
import logging
import threading
import time
import textwrap
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

import httpx as _httpx
from fastmcp import Client
from fastmcp.exceptions import ToolError


async def _call(tool: str, args: dict = None):
    """Call one MCP tool and return the parsed result.
    ToolErrors are caught and returned as {status: error, message: ...}
    rather than propagating — keeps cells running even when a tool fails.
    """
    try:
        async with Client(SERVER_URL) as client:
            raw = await client.call_tool(tool, args or {})
    except ToolError as e:
        return {'status': 'error', 'tool': tool, 'message': str(e)}
    except Exception as e:
        return {'status': 'error', 'tool': tool, 'message': f'{type(e).__name__}: {e}'}

    # FastMCP 3.x returns a CallToolResult object, not a list.
    content = raw.content if hasattr(raw, 'content') else (raw or [])
    if not content:
        return None
    item = content[0]
    text = item.text if hasattr(item, 'text') else str(item)
    try:
        return json.loads(text)
    except (json.JSONDecodeError, ValueError):
        return text


def run(tool: str, args: dict = None):
    """Synchronous wrapper — use in notebook cells."""
    result = asyncio.run(_call(tool, args))
    print(json.dumps(result, indent=2, default=str))
    return result


def skip_llm() -> bool:
    """Return True and print a notice when the LLM key is absent."""
    if not HAS_LLM_KEY:
        print('[SKIPPED] Set RADIANT_OLLAMA_API_KEY to run LLM cells.')
        return True
    return False


def assert_ok(result, field: str = None):
    """Raise AssertionError on error or missing field."""
    assert result is not None, 'Tool returned no result'
    if isinstance(result, dict):
        if result.get('status') == 'error':
            raise AssertionError(
                f'Tool error: {result.get("message", result)}\n'
                f'If this is a Redis connection error, see the note in the'
                f' Query section about fixing the mcp branch.'
            )
        if field:
            assert field in result, f'Expected field "{field}" missing: {result}'
    print('[OK]')


async def _wait_for_server(url: str, timeout: int = 120, interval: float = 1.0) -> bool:
    """Poll until the server accepts connections or timeout expires."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            async with _httpx.AsyncClient() as http:
                await http.get(
                    url, timeout=2.0,
                    headers={'Accept': 'application/json, text/event-stream'}
                )
                return True
        except (_httpx.ConnectError, _httpx.ConnectTimeout, _httpx.ReadTimeout):
            await asyncio.sleep(interval)
    return False


print('Helpers loaded  ✓')


Helpers loaded  ✓


## 5  Server startup

The LLM backend must use lazy initialization (no network calls in `__init__`) per the
spec. If the server does not reach `ready` within 30 s, the LLM backend is still
making an eager connection — fix it in the `mcp` branch before continuing.


In [5]:
import subprocess, time as _time

# ── Kill any process already holding the port ─────────────────────────────────
_port = int(os.environ['RADIANT_PORT'])
subprocess.run(['fuser', '-k', f'{_port}/tcp'], capture_output=True)
_time.sleep(1.0)

logging.basicConfig(stream=__import__('sys').stderr, level=logging.WARNING, force=True)

_server_ready = threading.Event()
_server_error  = None
_transport_used = None

def _run_server():
    global _server_error, _transport_used
    try:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        from radiant_rag_mcp.server import mcp
        print('Package      : radiant_rag_mcp  ✓')
        _server_ready.set()

        host = os.environ['RADIANT_HOST']
        port = int(os.environ['RADIANT_PORT'])

        # FastMCP changed the Streamable HTTP transport name across versions.
        # Try each in order; the first one that doesn't raise immediately wins.
        for _t in ['http', 'streamable-http']:
            try:
                _transport_used = _t
                mcp.run(transport=_t, host=host, port=port)
                return                # clean exit (server stopped normally)
            except Exception as _e:
                if 'Unknown transport' in str(_e) or 'unknown transport' in str(_e).lower():
                    continue          # this name isn't supported, try next
                raise                 # real error — stop trying
        raise RuntimeError("No supported HTTP transport found (tried: http, streamable-http)")

    except Exception as exc:
        _server_error = exc
        if not _server_ready.is_set():
            _server_ready.set()

_thread = threading.Thread(target=_run_server, name='radiant-mcp', daemon=True)
_thread.start()

if not _server_ready.wait(timeout=30):
    raise TimeoutError('Server thread did not signal ready within 30 s.')
if _server_error:
    raise _server_error

print(f'Waiting for server at {SERVER_URL} ...')
_deadline = time.time() + 90
while time.time() < _deadline:
    if _server_error:
        raise RuntimeError(f'Server thread failed: {_server_error}')
    if asyncio.run(_wait_for_server(SERVER_URL, timeout=5)):
        print(f'Server ready  ✓  →  {SERVER_URL}  (transport: {_transport_used})')
        break
    time.sleep(1)
else:
    raise TimeoutError(
        f'Server did not bind to {SERVER_URL} within 90 s.\n'
        f'Transport tried: {_transport_used}\n'
        f'Error (if any): {_server_error}'
    )


Package      : radiant_rag_mcp  ✓
Waiting for server at http://127.0.0.1:8765/mcp ...


╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │                  
                 │                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                FastMCP 3.2.3                                 │                  
                 │                            https://gofastmcp.com                             │                  
                 │                                                                              │                  
                 │                  🖥  Server:      radiant-rag, 3.2.3                          │                  
                 │                  🚀 Deploy free: https://horizon.prefect.io                  │                  
                 │                                                                              │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯

2026-04-13 18:54:56,411 - radiant_rag_mcp.app - INFO - Initializing Radiant RAG...
2026-04-13 18:54:56,413 - radiant_rag_mcp.llm.client - INFO - Using new backend configuration system
2026-04-13 18:54:56,419 - radiant_rag_mcp.llm.backends.factory - INFO - Creating LLM backend: type=ollama
2026-04-13 18:54:56,421 - radiant_rag_mcp.llm.backends.llm_backends - INFO - Configured OpenAI-compatible LLM backend: model=gemma4:31b-cloud, base_url=https://ollama.com/v1
2026-04-13 18:54:56,422 - radiant_rag_mcp.llm.backends.factory - INFO - Creating embedding backend: type=local
2026-04-13 18:54:56,424 - radiant_rag_mcp.llm.backends.embedding_backends - INFO - Initializing sentence-transformers embedding backend: sentence-transformers/all-MiniLM-L12-v2
The following layers were not sharded: encoder.layer.*.output.dense.bias, pooler.dense.weight, pooler.dense.bias, embeddings.word_embeddings.weight, encoder.layer.*.attention.self.key.bias, encoder.layer.*.attention.self.query.bias, encoder.layer.*

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-04-13 18:55:01,656 - radiant_rag_mcp.utils.cache - INFO - Initialized EmbeddingCache with max_size=10000 (true LRU)
2026-04-13 18:55:01,657 - radiant_rag_mcp.llm.backends.embedding_backends - INFO - Loaded sentence-transformers embedding model: sentence-transformers/all-MiniLM-L12-v2 (dim=384, device=cpu)
2026-04-13 18:55:01,658 - radiant_rag_mcp.llm.backends.factory - INFO - Creating reranking backend: type=local
2026-04-13 18:55:01,659 - radiant_rag_mcp.llm.backends.reranking_backends - INFO - Initializing cross-encoder reranking backend: cross-encoder/ms-marco-MiniLM-L12-v2
The following layers were not sharded: bert.encoder.layer.*.attention.self.value.bias, bert.encoder.layer.*.attention.self.query.bias, bert.encoder.layer.*.attention.output.dense.bias, bert.embeddings.LayerNorm.weight, bert.embeddings.position_embeddings.weight, bert.embeddings.word_embeddings.weight, bert.encoder.layer.*.intermediate.dense.weight, bert.encoder.layer.*.attention.self.key.weight, bert.encoder

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-04-13 18:55:03,174 - radiant_rag_mcp.llm.backends.reranking_backends - INFO - Loaded cross-encoder reranking model: cross-encoder/ms-marco-MiniLM-L12-v2 (device=cpu)
2026-04-13 18:55:03,180 - radiant_rag_mcp.storage.factory - INFO - Creating Chroma vector store
2026-04-13 18:55:03,396 - radiant_rag_mcp.storage.chroma_store - INFO - Initialized ChromaDB store at 'data/chroma_db' with collection 'radiant_docs'
2026-04-13 18:55:03,398 - radiant_rag_mcp.utils.conversation - INFO - ConversationStore: storage backend is 'chroma' — using in-memory storage
2026-04-13 18:55:03,415 - radiant_rag_mcp.utils.metrics_export - INFO - Prometheus metrics exporter initialized
2026-04-13 18:55:03,417 - PlanningAgent - INFO - [4926f4d9] [enabled=True] Initialized PlanningAgent
2026-04-13 18:55:03,418 - QueryDecompositionAgent - INFO - [69557240] [enabled=True] Initialized QueryDecompositionAgent
2026-04-13 18:55:03,419 - QueryRewriteAgent - INFO - [c1681702] [enabled=True] Initialized QueryRewriteAge

[04/13/26 18:55:03] INFO     Starting MCP server 'radiant-rag' with transport 'http' on            ]8;id=404700;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=486665;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py#299\299]8;;\
                             http://127.0.0.1:8765/mcp                                                             

INFO:     Started server process [3382]
INFO:     Waiting for application startup.
2026-04-13 18:55:03,560 - mcp.server.streamable_http_manager - INFO - StreamableHTTP session manager started
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8765 (Press CTRL+C to quit)
2026-04-13 18:55:04,120 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 5a1fb7e2ca4e477dbc341eaad7a75817


INFO:     127.0.0.1:49592 - "GET /mcp HTTP/1.1" 400 Bad Request
Server ready  ✓  →  http://127.0.0.1:8765/mcp  (transport: http)


## 6  Connection verification


In [6]:
async def _list_tools():
    async with Client(SERVER_URL) as client:
        return await client.list_tools()

tools = asyncio.run(_list_tools())
print(f'{len(tools)} tool(s) registered:\n')
for t in sorted(tools, key=lambda x: x.name):
    desc = (t.description or '').splitlines()[0]
    print(f'  {t.name:<24}  {desc}')

EXPECTED = {
    'search_knowledge', 'query_knowledge', 'simple_query', 'start_conversation',
    'ingest_documents', 'ingest_url', 'get_index_stats', 'clear_index',
    'rebuild_bm25', 'set_ingest_mode',
}
registered = {t.name for t in tools}
missing    = EXPECTED - registered
extra      = registered - EXPECTED

print()
if missing:
    print(f'[WARNING] Missing tools : {sorted(missing)}')
if extra:
    print(f'[NOTE]    Extra tools   : {sorted(extra)}')
if not missing:
    print('All 10 expected tools registered  ✓')


2026-04-13 18:55:21,793 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: c6691f70216249338b1c90fe9e431628


INFO:     127.0.0.1:42294 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:55:21,804 - mcp.client.streamable_http - INFO - Received session ID: c6691f70216249338b1c90fe9e431628
2026-04-13 18:55:21,809 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:42310 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:42312 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:42316 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:55:21,835 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 18:55:21,847 - mcp.server.streamable_http - INFO - Terminating session: c6691f70216249338b1c90fe9e431628


INFO:     127.0.0.1:42332 - "DELETE /mcp HTTP/1.1" 200 OK
10 tool(s) registered:

  clear_index               Delete all indexed documents from the knowledge base.
  get_index_stats           Return document counts and system health for the knowledge base.
  ingest_documents          Index local files or directories into the knowledge base.
  ingest_url                Index a URL or GitHub repository into the knowledge base.
  query_knowledge           Answer a question using the full agentic RAG pipeline.
  rebuild_bm25              Rebuild the BM25 sparse index from the current vector store contents.
  search_knowledge          Retrieve documents from the knowledge base without LLM generation.
  set_ingest_mode           Set the default ingestion storage mode for this server session.
  simple_query              Answer a question using a lightweight retrieval-and-synthesis pipeline.
  start_conversation        Create a new conversation ID for multi-turn query sessions.

All 10 expecte

## 7  Pre-ingestion state


In [7]:
print('=== get_index_stats (before ingestion) ===')
r = run('get_index_stats')
assert_ok(r)


2026-04-13 18:55:27,643 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 3f4ac5d6d18944e88650c0143cd6c770


=== get_index_stats (before ingestion) ===
INFO:     127.0.0.1:58056 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:55:27,653 - mcp.client.streamable_http - INFO - Received session ID: 3f4ac5d6d18944e88650c0143cd6c770
2026-04-13 18:55:27,656 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:58062 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58078 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58090 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:55:27,683 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 18:55:27,694 - radiant_rag_mcp.storage.bm25_index - INFO - Creating new BM25 index


INFO:     127.0.0.1:58098 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:55:27,711 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 18:55:27,725 - mcp.server.streamable_http - INFO - Terminating session: 3f4ac5d6d18944e88650c0143cd6c770


INFO:     127.0.0.1:58100 - "DELETE /mcp HTTP/1.1" 200 OK
{
  "health": {
    "redis": true,
    "vector_index": {
      "name": "radiant_docs",
      "backend": "chroma",
      "persist_directory": "./data/chroma_db",
      "document_count": 0,
      "distance_function": "cosine",
      "embedding_dimension": 384,
      "metadata": {
        "hnsw:space": "cosine"
      }
    },
    "bm25_index": {
      "document_count": 0,
      "unique_terms": 0,
      "avg_doc_length": 0.0,
      "k1": 1.5,
      "b": 0.75,
      "dirty": false,
      "needs_rebuild": true,
      "index_path": "data/bm25_index.json.gz",
      "storage_format": "json.gz (pending)"
    },
    "conversation": true
  },
  "metrics": {
    "run_count": 0,
    "average_latency_ms": null,
    "average_success_rate": 1.0,
    "history": []
  }
}
[OK]


In [8]:
# Verify set_ingest_mode: confirm default, switch to flat, restore.
print('=== set_ingest_mode: hierarchical=True ===')
r = run('set_ingest_mode', {'hierarchical': True})
assert_ok(r, 'hierarchical')
assert r['hierarchical'] is True

print('\n=== set_ingest_mode: hierarchical=False ===')
r = run('set_ingest_mode', {'hierarchical': False})
assert_ok(r, 'hierarchical')
assert r['hierarchical'] is False

print('\n=== set_ingest_mode: restore ===')
r = run('set_ingest_mode', {'hierarchical': True})
assert r['hierarchical'] is True
print('[OK] Restored to hierarchical.')


2026-04-13 18:55:35,035 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: bd98406d72cf4eb2b6e49bb3a8dad714


=== set_ingest_mode: hierarchical=True ===
INFO:     127.0.0.1:44712 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:55:35,043 - mcp.client.streamable_http - INFO - Received session ID: bd98406d72cf4eb2b6e49bb3a8dad714
2026-04-13 18:55:35,047 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:44716 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:44718 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:44730 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:55:35,074 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest


...

INFO:     127.0.0.1:44818 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:55:35,372 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 18:55:35,384 - mcp.server.streamable_http - INFO - Terminating session: e6c4b1c86a4b4362b35c5d4e3086623b


INFO:     127.0.0.1:44834 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 18:55:35,389 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "hierarchical": true,
  "message": "Ingest mode set to 'hierarchical'.  All subsequent ingest_documents calls will use hierarchical storage by default."
}
[OK] Restored to hierarchical.


## 8  Document ingestion


In [9]:
TEST_DIR = Path('/tmp/radiant_test_docs')
TEST_DIR.mkdir(parents=True, exist_ok=True)

docs = {
    'rag_overview.md': textwrap.dedent("""
        # Retrieval-Augmented Generation

        Retrieval-Augmented Generation (RAG) combines dense vector search with
        large language models to answer questions grounded in a document corpus.

        ## Key components
        - **Embedding model** — converts text chunks into dense vectors.
        - **Vector store** — indexes vectors for approximate nearest-neighbour search.
        - **BM25 index** — keyword-based sparse retrieval for exact term matching.
        - **Hybrid fusion** — Reciprocal Rank Fusion (RRF) combines dense and sparse results.
        - **LLM** — synthesises a grounded answer from retrieved context.
    """).strip(),

    'storage_backends.md': textwrap.dedent("""
        # Storage Backends

        Radiant RAG supports three vector storage backends.

        ## ChromaDB
        Default backend. Embedded, no external services required.
        Best for development, testing, and small corpora.

        ## Redis Stack
        Production backend. Requires Redis Stack via Docker or native install.
        Supports HNSW indexing, binary quantization, and sub-millisecond latency.
        Docker: docker run -d --name redis-stack -p 6379:6379 redis/redis-stack-server

        ## PostgreSQL with pgvector
        Enterprise backend. Requires PostgreSQL with the pgvector extension.
        ACID-compliant; integrates with existing PostgreSQL infrastructure.
        Docker: docker run -d --name pgvector -p 5432:5432 pgvector/pgvector:pg16
    """).strip(),

    'performance.txt': textwrap.dedent("""
        Performance optimisations in Radiant RAG

        Intelligent caching: LRU cache for embeddings and query results.
        Embedding cache hit rate: 30-50 percent for typical workloads.
        Query cache hit rate: 20-40 percent.
        Repeated queries: up to 93 percent latency reduction on cache hits.

        Parallel execution: dense and BM25 retrieval run concurrently.
        Hybrid retrieval is approximately 50 percent faster than sequential.

        Binary quantization: 10-20x faster retrieval, 3.5x memory reduction.
        Accuracy retention with binary quantization: 95-96 percent.
    """).strip(),
}

for fname, content in docs.items():
    (TEST_DIR / fname).write_text(content)
    print(f'  created  {TEST_DIR / fname}')


  created  /tmp/radiant_test_docs/rag_overview.md
  created  /tmp/radiant_test_docs/storage_backends.md
  created  /tmp/radiant_test_docs/performance.txt


In [10]:
print('=== ingest_documents (hierarchical=True) ===')
r = run('ingest_documents', {'paths': [str(TEST_DIR)], 'hierarchical': True})
assert_ok(r)


=== ingest_documents (hierarchical=True) ===


2026-04-13 18:55:51,809 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 64d30b7cb5764fc6a59ec14f25d1b90c


INFO:     127.0.0.1:53388 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:55:51,816 - mcp.client.streamable_http - INFO - Received session ID: 64d30b7cb5764fc6a59ec14f25d1b90c
2026-04-13 18:55:51,818 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:53400 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:53408 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:53416 - "POST /mcp HTTP/1.1" 200 OK


...

INFO:     127.0.0.1:35682 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:55:53,858 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 18:55:53,871 - mcp.server.streamable_http - INFO - Terminating session: 64d30b7cb5764fc6a59ec14f25d1b90c


INFO:     127.0.0.1:35692 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 18:55:53,877 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "files_processed": 3,
  "files_failed": 0,
  "chunks_created": 3,
  "documents_stored": 9,
  "errors": []
}
[OK]


In [11]:
%%time
# Fetch a single page — no_crawl=True avoids following links.
print('=== ingest_url (no_crawl=True) ===')
r = run('ingest_url', {
    'url': 'https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md',
    'no_crawl': True,
})
assert_ok(r)


2026-04-13 18:55:57,911 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 040d92c67db94700bdeff624e67f0ffb


=== ingest_url (no_crawl=True) ===
INFO:     127.0.0.1:35702 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:55:57,922 - mcp.client.streamable_http - INFO - Received session ID: 040d92c67db94700bdeff624e67f0ffb
2026-04-13 18:55:57,925 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:35710 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:35716 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:35720 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:55:57,960 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 18:55:57,965 - radiant_rag_mcp.ingestion.web_crawler - INFO - Starting crawl with 1 seed URLs, max_depth=0
2026-04-13 18:55:58,221 - radiant_rag_mcp.ingestion.web_crawler - INFO - Crawl complete: 1 pages crawled, 0 failed, 43317 bytes
2026-04-13 18:55:58,222 - radiant_rag_mcp.app - INFO - Crawled 1 pages, 0 failed, 43317 bytes
2026-04-13 18:56:07,005 - radiant_rag_mcp.storage.bm25_index - INFO - Syncing BM25 index with store...
2026-04-13 18:56:07,097 - radiant_rag_mcp.storage.bm25_index - INFO - Saved BM25 index with 76 documents
2026-04-13 18:56:07,099 - radiant_rag_mcp.storage.bm25_index - INFO - Sync complete: 70 added, 0 removed


INFO:     127.0.0.1:35072 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:56:07,118 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 18:56:07,133 - mcp.server.streamable_http - INFO - Terminating session: 040d92c67db94700bdeff624e67f0ffb


INFO:     127.0.0.1:35074 - "DELETE /mcp HTTP/1.1" 200 OK
{
  "urls_crawled": 1,
  "files_processed": 1,
  "files_failed": 0,
  "chunks_created": 70,
  "documents_stored": 140,
  "github_repos": 0,
  "errors": []
}
[OK]
CPU times: user 8.25 s, sys: 339 ms, total: 8.59 s
Wall time: 9.33 s


In [12]:
%%time
print('=== get_index_stats (after ingestion) ===')
r = run('get_index_stats')
assert_ok(r)
# Document count is nested: health.vector_index.document_count
total = (
    r.get('health', {}).get('vector_index', {}).get('document_count', 0)
    if isinstance(r, dict) else 0
)
assert total > 0, f'Expected documents > 0, got {total}'
print(f'[OK] {total} document(s) indexed.')


2026-04-13 18:56:12,631 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 585b85f470e64ec7a5424199064b2440


=== get_index_stats (after ingestion) ===
INFO:     127.0.0.1:35088 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:56:12,647 - mcp.client.streamable_http - INFO - Received session ID: 585b85f470e64ec7a5424199064b2440
2026-04-13 18:56:12,650 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:35096 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:35110 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:35116 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:56:12,672 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest


INFO:     127.0.0.1:35126 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:56:12,692 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 18:56:12,706 - mcp.server.streamable_http - INFO - Terminating session: 585b85f470e64ec7a5424199064b2440


INFO:     127.0.0.1:35132 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 18:56:12,710 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "health": {
    "redis": true,
    "vector_index": {
      "name": "radiant_docs",
      "backend": "chroma",
      "persist_directory": "./data/chroma_db",
      "document_count": 149,
      "distance_function": "cosine",
      "embedding_dimension": 384,
      "metadata": {
        "hnsw:space": "cosine"
      }
    },
    "bm25_index": {
      "document_count": 76,
      "unique_terms": 952,
      "avg_doc_length": 51.71052631578948,
      "k1": 1.5,
      "b": 0.75,
      "dirty": false,
      "needs_rebuild": true,
      "index_path": "data/bm25_index.json.gz",
      "storage_format": "json.gz"
    },
    "conversation": true
  },
  "metrics": {
    "run_count": 0,
    "average_latency_ms": null,
    "average_success_rate": 1.0,
    "history": []
  }
}
[OK]
[OK] 149 document(s) indexed.
CPU times: user 141 ms, sys: 10.9 ms, total: 151 ms
Wall time: 154 ms


## 9  Search  *(no LLM required)*


In [13]:
%%time
print('=== search_knowledge: mode=hybrid ===')
r = run('search_knowledge', {
    'query': 'hybrid retrieval BM25 vector search RRF',
    'mode': 'hybrid',
    'top_k': 3,
})
assert_ok(r)


2026-04-13 18:56:23,601 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 361a3d2a255e4b34a3d64327e6dd8380


=== search_knowledge: mode=hybrid ===
INFO:     127.0.0.1:57564 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:56:23,612 - mcp.client.streamable_http - INFO - Received session ID: 361a3d2a255e4b34a3d64327e6dd8380
2026-04-13 18:56:23,615 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:57568 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:57572 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57588 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:56:23,652 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 18:56:23,659 - DenseRetrievalAgent - INFO - [c12da012] [enabled=True] Initialized DenseRetrievalAgent
2026-04-13 18:56:23,722 - DenseRetrievalAgent - INFO - [650a0761] [query_length=39 num_results=3 top_k=3] Dense retrieval completed
2026-04-13 18:56:23,723 - DenseRetrievalAgent - INFO - [650a0761] [duration_ms=62.23 status=success] Execution completed
2026-04-13 18:56:23,725 - BM25RetrievalAgent - INFO - [38945156] [enabled=True] Initialized BM25RetrievalAgent
2026-04-13 18:56:23,743 - BM25RetrievalAgent - INFO - [7d151968] [query_length=39 num_results=3 top_k=3] BM25 retrieval completed
2026-04-13 18:56:23,745 - BM25RetrievalAgent - INFO - [7d151968] [duration_ms=19.10 status=success] Execution completed
2026-04-13 18:56:23,746 - RRFAgent - INFO - [2d8799c0] [enabled=True] Initialized RRFAgent
2026-04-13 18:56:23,748 - RRFAgent - INFO - [2aa8fdf9] [num_runs=2 total_docs

INFO:     127.0.0.1:57604 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:56:23,769 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 18:56:23,784 - mcp.server.streamable_http - INFO - Terminating session: 361a3d2a255e4b34a3d64327e6dd8380


INFO:     127.0.0.1:57616 - "DELETE /mcp HTTP/1.1" 200 OK
{
  "query": "hybrid retrieval BM25 vector search RRF",
  "mode": "hybrid",
  "results": [
    {
      "doc_id": "2b55bb2f254c42b5f4b21f2037ff4635eace49c3e0051491a0602e007d1c7560",
      "content": "# Retrieval-Augmented Generation Retrieval-Augmented Generation (RAG) combines dense vector search with large language models to answer questions grounded in a document corpus. ## Key components - **Embedding model** \u2014 converts text chunks into dense vectors. - **Vector store** \u2014 indexes vectors for approximate nearest-neighbour search. - **BM25 index** \u2014 keyword-based sparse retrieval for exact term matching. - **Hybrid fusion** \u2014 Reciprocal Rank Fusion (RRF) combines dense and sparse results. - **LLM** \u2014",
      "meta": {
        "cleaning": {
          "enabled": true,
          "bullets": false,
          "extra_whitespace": true,
          "dashes": false,
          "trailing_punctuation": false,
       

In [14]:
%%time
print('=== search_knowledge: mode=dense ===')
r = run('search_knowledge', {
    'query': 'storage backend production deployment comparison',
    'mode': 'dense',
    'top_k': 3,
})
assert_ok(r)


2026-04-13 18:56:35,463 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: d301c109c14342d19d2d6582a11c1037


=== search_knowledge: mode=dense ===
INFO:     127.0.0.1:45994 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:56:35,473 - mcp.client.streamable_http - INFO - Received session ID: d301c109c14342d19d2d6582a11c1037
2026-04-13 18:56:35,476 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:46000 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:46002 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:46010 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:56:35,513 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 18:56:35,516 - DenseRetrievalAgent - INFO - [424f36c0] [enabled=True] Initialized DenseRetrievalAgent
2026-04-13 18:56:35,578 - DenseRetrievalAgent - INFO - [f07150eb] [query_length=48 num_results=3 top_k=3] Dense retrieval completed
2026-04-13 18:56:35,580 - DenseRetrievalAgent - INFO - [f07150eb] [duration_ms=61.64 status=success] Execution completed


INFO:     127.0.0.1:46024 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:56:35,602 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 18:56:35,625 - mcp.server.streamable_http - INFO - Terminating session: d301c109c14342d19d2d6582a11c1037


INFO:     127.0.0.1:46030 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 18:56:35,633 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "query": "storage backend production deployment comparison",
  "mode": "dense",
  "results": [
    {
      "doc_id": "6eaabd8c6dbf957acd2aa4b32647e515bc21fc2ba6cbb6c2275a17aba4098434",
      "content": "# Storage Backends Radiant RAG supports three vector storage backends. ## ChromaDB Default backend. Embedded, no external services required. Best for development, testing, and small corpora. ## Redis Stack Production backend. Requires Redis Stack via Docker or native install. Supports HNSW indexing, binary quantization, and sub-millisecond latency. Docker: docker run -d --name redis-stack -p 6379:6379 redis/redis-stack-server ## PostgreSQL with pgvector Enterprise backend. Requires PostgreSQL with the",
      "meta": {
        "doc_level": "child",
        "child_total": 2,
        "child_index": 0,
        "has_embedding": true,
        "source_path": "/tmp/radiant_test_docs/storage_backends.md",
        "cleaning": {
          "enabled": true,
          "bullets": false,
         

In [15]:
%%time
print('=== search_knowledge: mode=bm25 ===')
r = run('search_knowledge', {
    'query': 'binary quantization latency memory reduction',
    'mode': 'bm25',
    'top_k': 3,
})
assert_ok(r)


2026-04-13 18:56:54,391 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: c08a59a099a242f1984fca45500f2c4a


=== search_knowledge: mode=bm25 ===
INFO:     127.0.0.1:46662 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:56:54,401 - mcp.client.streamable_http - INFO - Received session ID: c08a59a099a242f1984fca45500f2c4a
2026-04-13 18:56:54,404 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:46672 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:46676 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:46690 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:56:54,426 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 18:56:54,430 - BM25RetrievalAgent - INFO - [bc326bdf] [enabled=True] Initialized BM25RetrievalAgent
2026-04-13 18:56:54,437 - BM25RetrievalAgent - INFO - [7d253646] [query_length=44 num_results=3 top_k=3] BM25 retrieval completed
2026-04-13 18:56:54,438 - BM25RetrievalAgent - INFO - [7d253646] [duration_ms=7.15 status=success] Execution completed


INFO:     127.0.0.1:46698 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:56:54,454 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 18:56:54,466 - mcp.server.streamable_http - INFO - Terminating session: c08a59a099a242f1984fca45500f2c4a


INFO:     127.0.0.1:46710 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 18:56:54,473 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "query": "binary quantization latency memory reduction",
  "mode": "bm25",
  "results": [
    {
      "doc_id": "4c6caa7250787d3d02e2c713b95226f11cc3a2f28177ea9066d53f49b05e23b7",
      "content": "memory reduction. Accuracy retention with binary quantization: 95-96 percent.",
      "meta": {
        "child_index": 1,
        "doc_level": "child",
        "child_total": 2,
        "parent_id": "94877d7ea8b53f376d1919bca8bb95e02e9e83850c3e145d102df957ed6a99ee",
        "source_path": "/tmp/radiant_test_docs/performance.txt",
        "has_embedding": true,
        "cleaning": {
          "enabled": true,
          "bullets": false,
          "extra_whitespace": true,
          "dashes": false,
          "trailing_punctuation": false,
          "lowercase": false
        }
      },
      "score": 11.4283488746968
    },
    {
      "doc_id": "a313774c6cfbb31ea2e67a330c468e3592959445025abc151d9cf87ed778adb8",
      "content": "Performance optimisations in Radiant RAG Intelligent cachin

## 10  Query  `[LLM]`

In [16]:
%%time
if not skip_llm():
    print('=== simple_query ===')
    r = run('simple_query', {
        'question': 'What storage backend is best for production use?',
        'top_k': 5,
    })
    assert_ok(r, 'answer')


2026-04-13 18:57:07,648 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 8abeb11577194629aaf3095a952724f1


=== simple_query ===
INFO:     127.0.0.1:40228 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:57:07,657 - mcp.client.streamable_http - INFO - Received session ID: 8abeb11577194629aaf3095a952724f1
2026-04-13 18:57:07,659 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:40244 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:40258 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:40266 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:57:07,689 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 18:57:07,786 - radiant_rag_mcp.llm.backends.llm_backends - INFO - Initialized OpenAI-compatible LLM backend: model=gemma4:31b-cloud, base_url=https://ollama.com/v1


INFO:     127.0.0.1:40280 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:57:10,710 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 18:57:10,726 - mcp.server.streamable_http - INFO - Terminating session: 8abeb11577194629aaf3095a952724f1


INFO:     127.0.0.1:40294 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 18:57:10,731 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "question": "What storage backend is best for production use?",
  "answer": "Redis Stack is the production backend, as it supports HNSW indexing, binary quantization, and sub-millisecond latency."
}
[OK]
CPU times: user 1.23 s, sys: 93.9 ms, total: 1.33 s
Wall time: 3.15 s


In [17]:
%%time
if not skip_llm():
    print('=== query_knowledge (single-turn) ===')
    r = run('query_knowledge', {
        'question': 'What are the performance trade-offs between ChromaDB and Redis Stack?',
        'mode': 'hybrid',
        'conversation_id': None,
    })
    assert_ok(r, 'answer')


2026-04-13 18:57:24,338 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 55eb82184add4b10baf7118750ee947d


=== query_knowledge (single-turn) ===
INFO:     127.0.0.1:54294 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:57:24,347 - mcp.client.streamable_http - INFO - Received session ID: 55eb82184add4b10baf7118750ee947d
2026-04-13 18:57:24,349 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:54296 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:54310 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:54314 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 18:57:24,374 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 18:57:24,376 - radiant_rag_mcp.orchestrator - INFO - [8bee2997-3835-4660-824e-4d9e48a5d10a] Starting agentic pipeline for query: What are the performance trade-offs between ChromaDB and Redis Stack?...
2026-04-13 18:57:24,378 - radiant_rag_mcp.orchestrator - INFO - [8bee2997-3835-4660-824e-4d9e48a5d10a] Retrieval mode: hybrid, fast_path: False
2026-04-13 18:57:26,028 - QueryRewriteAgent - INFO - [8bee2997-3835-4660-824e-4d9e48a5d10a] [original=What are the performance trade-offs between Chroma rewritten=Compare the performance metrics and trade-offs bet] Query rewritten
2026-04-13 18:57:26,029 - QueryRewriteAgent - INFO - [8bee2997-3835-4660-824e-4d9e48a5d10a] [duration_ms=1648.96 status=success] Execution completed
2026-04-13 18:57:27,169 - QueryExpansionAgent - INFO - [8bee2997-3835-4660-824e-4d9e48a5d10a] [original=Compare the performance metrics and trade-offs bet num

INFO:     127.0.0.1:42686 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 19:05:04,093 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 19:05:04,112 - mcp.server.streamable_http - INFO - Terminating session: 55eb82184add4b10baf7118750ee947d


INFO:     127.0.0.1:42700 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 19:05:04,118 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "run_id": "8bee2997-3835-4660-824e-4d9e48a5d10a",
  "answer": "Based on the provided documents, the performance trade-offs between ChromaDB and Redis Stack are as follows:\n\n*   **Redis Stack:** Positioned as the production backend [DOC 1]. It is designed for low-latency, real-time use cases, offering sub-millisecond latency [DOC 1, DOC 6]. It supports high-performance features such as HNSW indexing and binary quantization [DOC 1]. However, it requires a Redis Stack installation via Docker or native install [DOC 1, DOC 6].\n*   **ChromaDB:** Positioned as the default backend for development, testing, and small corpora [DOC 1]. Its primary advantages are easy setup and being embedded, requiring no external services [DOC 1, DOC 4]. The trade-off is that it is less scalable than Redis [DOC 4].\n\n## Sources\n\n[1] **Source 1**\n    File: /tmp/radiant_test_docs/storage_backends.md\n[2] **Source 2**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[3] 

## 11  Multi-turn conversation  `[LLM]`


In [18]:
%%time
# start_conversation has no LLM dependency — always runs.
print('=== start_conversation ===')
conv = run('start_conversation', {})
assert_ok(conv, 'conversation_id')
conversation_id = conv['conversation_id']
print(f'\nconversation_id: {conversation_id}')


2026-04-13 19:09:49,535 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 1b9e9f00cf864e87a81e56205d376aea


=== start_conversation ===
INFO:     127.0.0.1:46258 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 19:09:49,544 - mcp.client.streamable_http - INFO - Received session ID: 1b9e9f00cf864e87a81e56205d376aea
2026-04-13 19:09:49,545 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:46264 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:46268 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:46284 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 19:09:49,567 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest


INFO:     127.0.0.1:46286 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 19:09:49,581 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 19:09:49,594 - mcp.server.streamable_http - INFO - Terminating session: 1b9e9f00cf864e87a81e56205d376aea


INFO:     127.0.0.1:46288 - "DELETE /mcp HTTP/1.1" 200 OK
{
  "conversation_id": "69e7925e-680c-48c6-8a27-305a7b65d3b1"
}
[OK]

conversation_id: 69e7925e-680c-48c6-8a27-305a7b65d3b1
CPU times: user 157 ms, sys: 1.67 ms, total: 158 ms
Wall time: 165 ms


In [19]:
%%time
if not skip_llm():
    print(f'=== query_knowledge: turn 1  (conv={conversation_id}) ===')
    r = run('query_knowledge', {
        'question': 'Which backend is best for a production deployment?',
        'mode': 'hybrid',
        'conversation_id': conversation_id,
    })
    assert_ok(r, 'answer')


2026-04-13 19:09:54,877 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: be9d737ba69144b4b119e2f671292370


=== query_knowledge: turn 1  (conv=69e7925e-680c-48c6-8a27-305a7b65d3b1) ===
INFO:     127.0.0.1:55016 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 19:09:54,885 - mcp.client.streamable_http - INFO - Received session ID: be9d737ba69144b4b119e2f671292370
2026-04-13 19:09:54,886 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:55030 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:55038 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55042 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 19:09:54,912 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 19:09:54,916 - radiant_rag_mcp.orchestrator - INFO - [201e96aa-a82f-4c6c-a74d-6c0932acfa09] Starting agentic pipeline for query: Which backend is best for a production deployment?...
2026-04-13 19:09:54,917 - radiant_rag_mcp.orchestrator - INFO - [201e96aa-a82f-4c6c-a74d-6c0932acfa09] Retrieval mode: hybrid, fast_path: True
2026-04-13 19:09:56,436 - QueryRewriteAgent - INFO - [201e96aa-a82f-4c6c-a74d-6c0932acfa09] [original=Which backend is best for a production deployment? rewritten=Comparison of industry-standard backend frameworks] Query rewritten
2026-04-13 19:09:56,437 - QueryRewriteAgent - INFO - [201e96aa-a82f-4c6c-a74d-6c0932acfa09] [duration_ms=1518.40 status=success] Execution completed
2026-04-13 19:09:56,531 - RRFAgent - INFO - [201e96aa-a82f-4c6c-a74d-6c0932acfa09] [num_runs=2 total_docs=16 fused_results=15] RRF fusion completed
2026-04-13 19:09:56,532 - RRFA

INFO:     127.0.0.1:56356 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 19:12:17,294 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 19:12:17,308 - mcp.server.streamable_http - INFO - Terminating session: be9d737ba69144b4b119e2f671292370


INFO:     127.0.0.1:56358 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 19:12:17,315 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "run_id": "201e96aa-a82f-4c6c-a74d-6c0932acfa09",
  "answer": "Based on the provided documents, the best backend for a production deployment is **Redis Stack**, which is explicitly described as the \"Production backend\" [DOC 1] and listed for the \"Production, low-latency\" use case [DOC 2]. It offers sub-millisecond latency and supports binary quantization and HNSW indexing [DOC 1].\n\nAdditionally, **PostgreSQL with pgvector** is identified as an \"Enterprise backend\" and is suitable for those who need ACID compliance or integration with existing PostgreSQL infrastructure [DOC 1].\n\n## Sources\n\n[1] **Source 1**\n    File: /tmp/radiant_test_docs/storage_backends.md\n[2] **Source 2**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[3] **Source 3**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[4] **Source 4**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[5] **Source 5*

In [20]:
%%time
if not skip_llm():
    print(f'=== query_knowledge: turn 2 follow-up  (conv={conversation_id}) ===')
    r = run('query_knowledge', {
        'question': 'What Docker command do I need to run for that backend?',
        'mode': 'hybrid',
        'conversation_id': conversation_id,
    })
    assert_ok(r, 'answer')


2026-04-13 19:15:27,284 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 1740119a4d6f478f8649c6b76e956f0c


=== query_knowledge: turn 2 follow-up  (conv=69e7925e-680c-48c6-8a27-305a7b65d3b1) ===
INFO:     127.0.0.1:34628 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 19:15:27,295 - mcp.client.streamable_http - INFO - Received session ID: 1740119a4d6f478f8649c6b76e956f0c
2026-04-13 19:15:27,299 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:34632 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:34636 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:34646 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 19:15:27,326 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 19:15:27,332 - radiant_rag_mcp.orchestrator - INFO - [5f97080b-bb07-462d-bf56-4b2b27f87ce7] Starting agentic pipeline for query: What Docker command do I need to run for that backend?...
2026-04-13 19:15:27,333 - radiant_rag_mcp.orchestrator - INFO - [5f97080b-bb07-462d-bf56-4b2b27f87ce7] Retrieval mode: hybrid, fast_path: False
2026-04-13 19:17:27,377 - openai._base_client - INFO - Retrying request to /chat/completions in 0.446074 seconds
2026-04-13 19:19:27,957 - openai._base_client - INFO - Retrying request to /chat/completions in 0.903875 seconds
2026-04-13 19:21:28,979 - radiant_rag_mcp.llm.backends.llm_backends - ERROR - OpenAI-compatible LLM request failed: Request timed out.
2026-04-13 19:21:28,980 - radiant_rag_mcp.llm.client - ERROR - LLM backend request failed: Request timed out.
2026-04-13 19:21:28,981 - QueryRewriteAgent - INFO - [5f97080b-bb07-462d-bf56-4b2b

INFO:     127.0.0.1:58022 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 19:28:20,064 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 19:28:20,077 - mcp.server.streamable_http - INFO - Terminating session: 1740119a4d6f478f8649c6b76e956f0c


INFO:     127.0.0.1:58028 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 19:28:20,082 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "run_id": "5f97080b-bb07-462d-bf56-4b2b27f87ce7",
  "answer": "To run the production backend (Redis Stack), use the following Docker command:\n\n`docker run -d --name redis-stack -p 6379:6379 redis/redis-stack-server`\n\nThis command is cited in [DOC 1], [DOC 3], and [DOC 4].\n\n## Sources\n\n[1] **Source 1**\n    File: /tmp/radiant_test_docs/storage_backends.md\n[2] **Source 2**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[3] **Source 3**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[4] **Source 4**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[5] **Source 5**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[6] **Source 6**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[7] **Source 7**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[8] **Source 8**\n    URL: htt

## 12  Maintenance


In [21]:
print('=== rebuild_bm25 ===')
r = run('rebuild_bm25', {})
assert_ok(r)

print('\n=== get_index_stats (after rebuild) ===')
run('get_index_stats')


2026-04-13 19:32:54,709 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: f94336ea7e3547829e0072eff52767f5


=== rebuild_bm25 ===
INFO:     127.0.0.1:33806 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 19:32:54,722 - mcp.client.streamable_http - INFO - Received session ID: f94336ea7e3547829e0072eff52767f5
2026-04-13 19:32:54,723 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:33812 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:33826 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:33828 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 19:32:54,749 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 19:32:54,755 - radiant_rag_mcp.storage.bm25_index - INFO - Building BM25 index from store (max 100000 documents)...
2026-04-13 19:32:54,860 - radiant_rag_mcp.storage.bm25_index - INFO - Saved BM25 index with 76 documents
2026-04-13 19:32:54,861 - radiant_rag_mcp.storage.bm25_index - INFO - Built BM25 index with 76 documents


...

2026-04-13 19:32:55,001 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest


INFO:     127.0.0.1:33894 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 19:32:55,031 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 19:32:55,047 - mcp.server.streamable_http - INFO - Terminating session: 415034dad1414fb49b4765f41d58bddc


INFO:     127.0.0.1:33898 - "DELETE /mcp HTTP/1.1" 200 OK
{
  "health": {
    "redis": true,
    "vector_index": {
      "name": "radiant_docs",
      "backend": "chroma",
      "persist_directory": "./data/chroma_db",
      "document_count": 149,
      "distance_function": "cosine",
      "embedding_dimension": 384,
      "metadata": {
        "hnsw:space": "cosine"
      }
    },
    "bm25_index": {
      "document_count": 76,
      "unique_terms": 952,
      "avg_doc_length": 51.71052631578949,
      "k1": 1.5,
      "b": 0.75,
      "dirty": false,
      "needs_rebuild": true,
      "index_path": "data/bm25_index.json.gz",
      "storage_format": "json.gz"
    },
    "conversation": true
  },
  "metrics": {
    "run_count": 3,
    "average_latency_ms": 458252.5538603465,
    "average_success_rate": 1.0,
    "history": [
      {
        "run_id": "8bee2997-3835-4660-824e-4d9e48a5d10a",
        "started_at": 1776106644.376593,
        "ended_at": 1776107104.071937,
        "total_lat

{'health': {'redis': True,
  'vector_index': {'name': 'radiant_docs',
   'backend': 'chroma',
   'persist_directory': './data/chroma_db',
   'document_count': 149,
   'distance_function': 'cosine',
   'embedding_dimension': 384,
   'metadata': {'hnsw:space': 'cosine'}},
  'bm25_index': {'document_count': 76,
   'unique_terms': 952,
   'avg_doc_length': 51.71052631578949,
   'k1': 1.5,
   'b': 0.75,
   'dirty': False,
   'needs_rebuild': True,
   'index_path': 'data/bm25_index.json.gz',
   'storage_format': 'json.gz'},
  'conversation': True},
 'metrics': {'run_count': 3,
  'average_latency_ms': 458252.5538603465,
  'average_success_rate': 1.0,
  'history': [{'run_id': '8bee2997-3835-4660-824e-4d9e48a5d10a',
    'started_at': 1776106644.376593,
    'ended_at': 1776107104.071937,
    'total_latency_ms': 459695.34397125244,
    'success_rate': 1.0,
    'steps': [{'name': 'QueryRewriteAgent',
      'started_at': 1776106644.38074,
      'ended_at': 1776106646.0307605,
      'latency_ms': 16

In [22]:
# Dry run — confirm=False must return status=cancelled, no data deleted.
print('=== clear_index: dry run (confirm=False) ===')
r = run('clear_index', {'confirm': False})
assert isinstance(r, dict) and r.get('cleared') is False, \
    f'Expected cleared=False, got: {r}'
print('[OK] Cancellation returned as expected — no data deleted.')


2026-04-13 19:33:01,613 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 342bb090a7c04343868166ee6c989f4e


=== clear_index: dry run (confirm=False) ===
INFO:     127.0.0.1:33906 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 19:33:01,622 - mcp.client.streamable_http - INFO - Received session ID: 342bb090a7c04343868166ee6c989f4e
2026-04-13 19:33:01,624 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:33910 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:33912 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:33920 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 19:33:01,649 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest


INFO:     127.0.0.1:33934 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 19:33:01,665 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 19:33:01,679 - mcp.server.streamable_http - INFO - Terminating session: 342bb090a7c04343868166ee6c989f4e


INFO:     127.0.0.1:33940 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 19:33:01,684 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "cleared": false,
  "message": "Index was NOT cleared.  Pass confirm=True to proceed.  This operation deletes all indexed documents and cannot be undone."
}
[OK] Cancellation returned as expected — no data deleted.


In [23]:
# WARNING: deletes all indexed documents.
# Comment this cell out to keep the test corpus.
print('=== clear_index: confirmed ===')
r = run('clear_index', {'confirm': True})
assert_ok(r)

print('\n=== get_index_stats (after clear) ===')
r = run('get_index_stats')
total = (
    r.get('health', {}).get('vector_index', {}).get('document_count', -1)
    if isinstance(r, dict) else -1
)
assert total == 0, f'Expected 0 documents after clear, got {total}'
print('[OK] Index empty.')


2026-04-13 19:33:10,386 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 7f88d84a987942139ba08b78a2e73da4


=== clear_index: confirmed ===
INFO:     127.0.0.1:44028 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 19:33:10,398 - mcp.client.streamable_http - INFO - Received session ID: 7f88d84a987942139ba08b78a2e73da4
2026-04-13 19:33:10,401 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:44030 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:44044 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:44054 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 19:33:10,427 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 19:33:10,511 - radiant_rag_mcp.storage.chroma_store - INFO - Dropped Chroma collection 'radiant_docs'
2026-04-13 19:33:10,513 - radiant_rag_mcp.app - ERROR - Failed to clear index: 'ChromaVectorStore' object has no attribute '_ensure_index'


...

INFO:     127.0.0.1:44098 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 19:33:10,706 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 19:33:10,721 - mcp.server.streamable_http - INFO - Terminating session: 84fa8ee024e146b0a88700cec4155ee6


INFO:     127.0.0.1:44100 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 19:33:10,728 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "health": {
    "redis": true,
    "vector_index": {
      "name": "radiant_docs",
      "backend": "chroma",
      "persist_directory": "./data/chroma_db",
      "document_count": 0,
      "distance_function": "cosine",
      "embedding_dimension": 384,
      "metadata": {
        "hnsw:space": "cosine"
      }
    },
    "bm25_index": {
      "document_count": 76,
      "unique_terms": 952,
      "avg_doc_length": 51.71052631578949,
      "k1": 1.5,
      "b": 0.75,
      "dirty": false,
      "needs_rebuild": true,
      "index_path": "data/bm25_index.json.gz",
      "storage_format": "json.gz"
    },
    "conversation": true
  },
  "metrics": {
    "run_count": 3,
    "average_latency_ms": 458252.5538603465,
    "average_success_rate": 1.0,
    "history": [
      {
        "run_id": "8bee2997-3835-4660-824e-4d9e48a5d10a",
        "started_at": 1776106644.376593,
        "ended_at": 1776107104.071937,
        "total_latency_ms": 459695.34397125244,
        "success_rate": 1.0,
 

## 13  Summary


In [24]:
print('=' * 55)
print('  Test run complete')
print('=' * 55)
print(f'  Branch        : {ACTIVE_BRANCH}')
print(f'  Server URL    : {SERVER_URL}')
print(f'  Backend       : {os.environ["RADIANT_STORAGE_BACKEND"]}')
print(f'  LLM cells ran : {HAS_LLM_KEY}')
print(f'  Test docs     : {TEST_DIR}')
print()
print('The server thread is daemonised and stops when the runtime ends.')
print('Re-run the server startup cell to restart it in the same session.')


  Test run complete
  Branch        : mcp
  Server URL    : http://127.0.0.1:8765/mcp
  Backend       : chroma
  LLM cells ran : True
  Test docs     : /tmp/radiant_test_docs

The server thread is daemonised and stops when the runtime ends.
Re-run the server startup cell to restart it in the same session.
